In [ ]:
# ── MLOps bootstrap (auto-injected by inject_mlops_cell.py) ──────────────────
import os, warnings, mlflow
warnings.filterwarnings("ignore")

SEED = 42
import random, numpy as np
random.seed(SEED)
np.random.seed(SEED)
try:
    import torch; torch.manual_seed(SEED)
except ImportError:
    pass
try:
    import tensorflow as tf; tf.random.set_seed(SEED)
except ImportError:
    pass

_nb_name = os.path.basename(os.path.abspath("__file__") if "__file__" in dir() else ".").replace(".ipynb","")
mlflow.set_tracking_uri("sqlite:///" + str(Path(__file__).parent.parent.parent / "mlflow.db")
                        if "__file__" in dir() else "sqlite:///mlflow.db")
_exp = mlflow.set_experiment(_nb_name or "unnamed_notebook")
print(f"MLflow experiment: {_exp.name}")


# Content Repurposer — One Article, Many Formats

## 1. Project Overview

**Task:** Take one long-form blog post or article and repurpose it into multiple social media and communication formats — LinkedIn post, Twitter/X thread, email snippet, short summary — with controllable tone and style.

**Approach:** Prompt adaptation — the same source content is fed through different format-specific prompts, each with platform-aware constraints (character limits, tone, structure).

**Stack:**
- `ChatOllama` + `qwen3.5:9b` — local LLM for all generation
- `LangChain` — message formatting
- Pure Python — no external dependencies beyond LangChain

> **No API keys required.** Everything runs locally with Ollama.

## 2. Learning Goals

| # | Skill |
|---|-------|
| 1 | **Adapt prompts** for different output formats from identical source material |
| 2 | **Enforce platform constraints** — character limits, thread structure, hashtags |
| 3 | **Control tone and style** — professional, casual, persuasive, educational |
| 4 | **Structure a Twitter/X thread** — hook → points → CTA across multiple tweets |
| 5 | **Write LinkedIn-native content** — storytelling format, emojis, line breaks |
| 6 | **Draft email snippets** — subject line + preview + body with clear CTA |
| 7 | **Compare outputs** — evaluate the same content across formats and tones |

## 3. Problem Statement

Content creators, marketers, and founders spend hours repurposing a single blog post into social media content. A 2,000-word article could become:
- A LinkedIn post (1,300 chars optimal)
- A Twitter/X thread (5-8 tweets, 280 chars each)
- An email newsletter snippet (150-word teaser)
- A one-paragraph summary for a Slack channel

Each format has different rules: tone, length, structure, audience expectations. Manually rewriting is tedious and inconsistent.

## 4. Why This Project Matters

- **Prompt engineering showcase** — same content, different prompts → dramatically different outputs
- **Real business value** — content repurposing tools (Repurpose.io, Lately.ai) charge $50-200/mo
- **Tone control** is the hardest prompt engineering skill — this project practices it extensively
- **Format constraints** teach you to guide LLMs within strict boundaries (char limits, structure)

## 5. Pipeline Architecture

```
Long-form Article / Blog Post
            │
            ▼
    [Extract Key Points] ── identify main ideas, quotes, stats
            │
            ├──▶ [LinkedIn Post]     — storytelling, professional, 1300 chars
            │
            ├──▶ [Twitter/X Thread]  — hook + 5-8 tweets + CTA, 280/tweet
            │
            ├──▶ [Email Snippet]     — subject + preview + body + CTA
            │
            ├──▶ [Short Summary]     — 2-3 sentences, neutral
            │
            └──▶ [Tone Variations]   — same format, different styles
```

### Why Separate Prompts?

A single "convert to social media" prompt produces generic, platform-unaware content. Each platform has its own:
- **Length limits** (Twitter: 280 chars, LinkedIn: ~3000 chars)
- **Structural norms** (threads vs paragraphs vs bullet points)
- **Tone expectations** (LinkedIn: professional, Twitter: punchy, Email: direct)
- **Engagement patterns** (hooks, CTAs, hashtags)

## 6. Environment Setup

In [ ]:
# Uncomment if any package is missing
# !pip install -q langchain langchain-ollama langchain-core

print("Dependencies: langchain, langchain-ollama")
print("LLM: Ollama with qwen3.5:9b (must be running locally)")

## 7. Imports

In [ ]:
import os
import re
import json
import textwrap
import warnings

os.environ["USE_TF"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
warnings.filterwarnings("ignore")

from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage, SystemMessage

print("All imports OK")

## 8. Configuration

In [ ]:
LLM_MODEL = "qwen3.5:9b"

TEMP_EXTRACT = 0.0    # Key-point extraction — deterministic
TEMP_GENERATE = 0.5   # Content generation — creative
TEMP_FORMAL = 0.3     # Formal / professional tone
TEMP_CASUAL = 0.7     # Casual / conversational tone

print("Configuration loaded")
print(f"  LLM: {LLM_MODEL}")
print(f"  Temperatures: extract={TEMP_EXTRACT}, generate={TEMP_GENERATE}, "
      f"formal={TEMP_FORMAL}, casual={TEMP_CASUAL}")

## 9. Helper Functions

In [ ]:
def clean(text: str) -> str:
    """Strip thinking tags from LLM output."""
    if "<think>" in text:
        text = text.split("</think>")[-1].strip()
    return text.strip()


def parse_json(text: str):
    """Extract JSON from LLM output."""
    text = clean(text)
    if "```" in text:
        text = re.sub(r"```(?:json)?\s*", "", text)
        text = text.replace("```", "")
    start = text.find("{")
    alt = text.find("[")
    if alt >= 0 and (start < 0 or alt < start):
        start = alt
    end = max(text.rfind("}"), text.rfind("]")) + 1
    if start >= 0 and end > start:
        try:
            return json.loads(text[start:end])
        except json.JSONDecodeError:
            pass
    return None


def ask(prompt: str, system: str = "", temperature: float = 0.5) -> str:
    """Send a prompt to the LLM and return cleaned text."""
    llm = ChatOllama(model=LLM_MODEL, temperature=temperature)
    messages = []
    if system:
        messages.append(SystemMessage(content=system))
    messages.append(HumanMessage(content=prompt))
    resp = llm.invoke(messages)
    return clean(resp.content)


def wrap_print(text: str, width: int = 80):
    """Print text with word wrapping."""
    for line in text.split("\n"):
        if line.strip():
            print(textwrap.fill(line, width=width))
        else:
            print()


# Quick LLM test
test = ask("Say 'ready'.")
print(f"LLM ready: {test[:30]}")

---

# Part A — Source Article

## 10. The Blog Post

We use a realistic tech blog post as our source material. This is the kind of content a startup founder or marketer would want to repurpose across platforms.

In [ ]:
ARTICLE = """Title: Why We Moved Our Entire Infrastructure to Edge Computing — And What We Learned

Six months ago, our startup served 50,000 API requests per day from a single AWS region in
us-east-1. Our p99 latency for users in Asia was 340ms. For a real-time collaboration tool,
that's unacceptable — every 100ms of delay reduces user engagement by 7%. We decided to go
all-in on edge computing, deploying our backend to 42 edge locations across 6 continents.
Here's what happened.

The Problem: Centralized = Slow for Everyone Except You

When your servers sit in Virginia, users in Virginia get 12ms latency. Users in Tokyo get
340ms. Users in São Paulo get 280ms. We were building a "global" product that only felt fast
in North America. Our user surveys confirmed it: 62% of international users rated our app as
"sluggish" compared to 8% of US users. The irony? International users were our
fastest-growing segment — 73% of new signups came from outside the US.

What We Did: Edge-First Architecture

We rebuilt our API layer to run on Cloudflare Workers, deployed our database reads through a
globally replicated read-replica setup using CockroachDB, and moved static assets to a CDN
with 280+ PoPs. The key architectural decisions:

1. Stateless API handlers run at the edge (42 locations). Each request is processed by the
nearest edge node. No central routing.

2. Database writes still go to a primary region (us-east-1), but we accept the write latency
trade-off because 94% of our requests are reads. Writes use async acknowledgment so users
don't wait for cross-region replication.

3. Session state is stored in a distributed KV store (Cloudflare KV) with eventual
consistency. We designed our app to tolerate 2-second consistency windows.

4. Feature flags and config are pushed to the edge every 30 seconds via a global broadcast
channel. No per-request config fetches.

The Results: Night and Day

After 3 months of edge deployment:
- Global p99 latency dropped from 340ms to 47ms (86% reduction)
- International user engagement increased 34%
- Page load times improved 2.8x for users outside North America
- Our monthly AWS bill decreased 23% despite serving 3x more traffic
- Uptime improved from 99.92% to 99.99% due to multi-region failover

The engagement numbers were the real surprise. We expected latency improvement but didn't
anticipate that faster response times would increase feature adoption. Users who previously
avoided real-time features (collaborative editing, live cursors) started using them heavily
once the lag disappeared.

What We Got Wrong

It wasn't all smooth. Three lessons we learned the hard way:

First, debugging distributed systems is 10x harder than monoliths. When a bug happens at the
edge in Mumbai but not in London, you need distributed tracing from day one. We didn't have
it and lost a week tracking down a timezone-related serialization bug.

Second, eventual consistency breaks user expectations in subtle ways. A user would update
their profile, refresh the page, and see the old data because the read hit a different
replica. We had to implement read-your-own-writes consistency for user-facing mutations.

Third, edge computing doesn't eliminate the need for a primary region. Critical operations —
billing, auth token generation, data exports — still need strong consistency and should run
in one place. Trying to distribute everything was a mistake that cost us two weeks.

Should You Move to the Edge?

Not always. Edge computing makes sense if: (a) you have a global user base, (b) latency
directly impacts your core UX, and (c) most of your traffic is reads, not writes. If your
users are in one region and your app is write-heavy, a well-configured single-region setup
with a CDN for static assets is simpler, cheaper, and easier to debug.

For us, the move was transformative. Our NPS score among international users jumped from +12
to +51, and our 30-day retention improved by 18 percentage points. The edge isn't just about
speed — it's about making your product feel native everywhere in the world."""

print(f"Article loaded: {len(ARTICLE):,} characters")
print(f"Word count: ~{len(ARTICLE.split()):,}")
print(f"Title: {ARTICLE.split(chr(10))[0].replace('Title: ', '')}")

## 11. Extract Key Points

Before generating any content, extract the article's key points, statistics, and quotable moments. This structured extraction feeds into all downstream formats.

In [ ]:
EXTRACT_PROMPT = """Analyze this blog post and extract structured information.

Article:
{article}

Return ONLY JSON:
{{
  "title": "article title",
  "main_thesis": "one-sentence summary of the article's argument",
  "key_points": ["point 1", "point 2", "point 3", "point 4", "point 5"],
  "statistics": ["stat 1", "stat 2", "stat 3"],
  "quotable_lines": ["compelling line 1", "compelling line 2"],
  "target_audience": "who this is written for",
  "call_to_action": "what the author wants the reader to do/think"
}}"""

extract_resp = ask(EXTRACT_PROMPT.format(article=ARTICLE), temperature=TEMP_EXTRACT)
key_points = parse_json(extract_resp)

print("EXTRACTED KEY POINTS")
print("=" * 60)

if key_points:
    print(f"\nThesis: {key_points.get('main_thesis', '')}")
    print(f"Audience: {key_points.get('target_audience', '')}")
    print(f"\nKey Points:")
    for p in key_points.get("key_points", []):
        print(f"  • {p}")
    print(f"\nStatistics:")
    for s in key_points.get("statistics", []):
        print(f"  📊 {s}")
    print(f"\nQuotable Lines:")
    for q in key_points.get("quotable_lines", []):
        print(f"  \" {q}\"")
else:
    print("(parse error)")
    print(extract_resp[:300])

---

# Part B — Format-Specific Generation

## 12. LinkedIn Post

### LinkedIn format norms
- **Hook** in the first 2 lines (before "see more")
- Short paragraphs (1-2 sentences each)
- Strategic line breaks for readability
- Professional but personal tone
- End with a question or CTA to drive comments
- 1,300 characters is the sweet spot (before engagement drops)
- Hashtags at the end (3-5 relevant ones)

In [ ]:
LINKEDIN_SYSTEM = """You write LinkedIn posts that get high engagement.

LinkedIn post rules:
- First 2 lines are the HOOK (this shows before 'see more'). Make them compelling.
- Use short paragraphs (1-2 sentences each).
- Include ONE concrete number or statistic.
- Use line breaks between paragraphs for mobile readability.
- End with a question that invites comments.
- Add 3-5 relevant hashtags at the very end.
- Total length: 1000-1500 characters.
- Tone: professional but human. First person. Share a lesson learned."""

LINKEDIN_PROMPT = """Convert this blog post into a LinkedIn post.

Article:
{article}

Write the LinkedIn post (1000-1500 chars):"""

linkedin_post = ask(
    LINKEDIN_PROMPT.format(article=ARTICLE),
    system=LINKEDIN_SYSTEM,
    temperature=TEMP_GENERATE,
)

print("LINKEDIN POST")
print("=" * 60)
print(linkedin_post)
print(f"\n{'─'*60}")
print(f"Characters: {len(linkedin_post):,} ", end="")
if len(linkedin_post) <= 1500:
    print("✅ Within 1500-char sweet spot")
else:
    print(f"⚠ Over limit by {len(linkedin_post) - 1500} chars")

## 13. Twitter/X Thread

### Twitter thread format norms
- Tweet 1: **hook** — must stand alone and grab attention
- Body tweets: one idea per tweet, numbered
- Last tweet: takeaway + CTA (follow, retweet, bookmark)
- Each tweet: ≤ 280 characters
- 5-8 tweets is the optimal length
- Use 🧵 emoji on the first tweet

In [ ]:
TWITTER_SYSTEM = """You write Twitter/X threads that go viral.

Thread rules:
- Tweet 1: Attention-grabbing hook. Use a surprising stat or bold claim.
  End with 🧵 emoji.
- Tweets 2-6: One key point per tweet. Use numbers, short sentences.
- Last tweet: Key takeaway + CTA (bookmark/retweet/follow).
- EVERY tweet must be ≤ 280 characters. This is NON-NEGOTIABLE.
- Total: 5-8 tweets.
- Tone: punchy, direct, sometimes provocative. No filler words."""

TWITTER_PROMPT = """Convert this article into a Twitter/X thread of 6-8 tweets.

Article:
{article}

Return ONLY JSON array of tweets:
["tweet 1 text", "tweet 2 text", ...]"""

thread_resp = ask(
    TWITTER_PROMPT.format(article=ARTICLE),
    system=TWITTER_SYSTEM,
    temperature=TEMP_GENERATE,
)
thread = parse_json(thread_resp)

print("TWITTER/X THREAD")
print("=" * 60)

if thread and isinstance(thread, list):
    all_ok = True
    for i, tweet in enumerate(thread, 1):
        length = len(tweet)
        ok = "✅" if length <= 280 else "⚠️"
        if length > 280:
            all_ok = False
        print(f"\n  [{i}/{len(thread)}] ({length} chars) {ok}")
        print(f"  {tweet}")
    print(f"\n{'─'*60}")
    print(f"Total tweets: {len(thread)}")
    print(f"All under 280 chars: {'✅ Yes' if all_ok else '⚠ Some over limit'}")
else:
    print("(JSON parse error — raw response below)")
    wrap_print(thread_resp[:500])

## 14. Email Newsletter Snippet

### Email format norms
- **Subject line**: 6-10 words, curiosity-driven or value-driven
- **Preview text**: 40-90 chars (shows in inbox alongside subject)
- **Body**: 100-150 words, teases the article, doesn't reproduce it
- **CTA**: one clear action ("Read the full post →")

In [ ]:
EMAIL_SYSTEM = """You write email newsletter snippets. Your goal is to get readers to
click through and read the full article.

Email rules:
- Subject line: 6-10 words, creates curiosity or promises value.
- Preview text: 40-90 characters, complements the subject.
- Body: 100-150 words. Tease the key insight WITHOUT giving everything away.
- Include ONE specific stat to add credibility.
- End with a clear CTA: 'Read the full post →'
- Tone: conversational, like writing to a smart friend."""

EMAIL_PROMPT = """Convert this article into an email newsletter snippet.

Article:
{article}

Return ONLY JSON:
{{
  "subject_line": "...",
  "preview_text": "...",
  "body": "...",
  "cta": "Read the full post →"
}}"""

email_resp = ask(
    EMAIL_PROMPT.format(article=ARTICLE),
    system=EMAIL_SYSTEM,
    temperature=TEMP_GENERATE,
)
email = parse_json(email_resp)

print("EMAIL NEWSLETTER SNIPPET")
print("=" * 60)

if email:
    print(f"\n  Subject: {email.get('subject_line', '')}")
    print(f"  Preview: {email.get('preview_text', '')}")
    print(f"\n{'─'*60}")
    wrap_print(email.get("body", ""))
    print(f"\n  [{email.get('cta', 'Read more →')}]")
    body_words = len(email.get("body", "").split())
    print(f"\n{'─'*60}")
    print(f"  Subject length: {len(email.get('subject_line',''))} chars")
    print(f"  Preview length: {len(email.get('preview_text',''))} chars", end=" ")
    plen = len(email.get("preview_text", ""))
    print("✅" if 40 <= plen <= 90 else f"⚠ target: 40-90")
    print(f"  Body: {body_words} words", end=" ")
    print("✅" if 80 <= body_words <= 180 else f"⚠ target: 100-150")
else:
    print("(parse error)")
    wrap_print(email_resp[:300])

## 15. Short Summary

A neutral, informational summary for Slack channels, README files, or link previews. 2-3 sentences, no marketing language.

In [ ]:
SUMMARY_PROMPT = """Write a 2-3 sentence neutral summary of this article.
No marketing language, no exclamation marks, no superlatives.
Just state what the article covers and the key finding.

Article:
{article}

Summary (2-3 sentences):"""

short_summary = ask(
    SUMMARY_PROMPT.format(article=ARTICLE),
    temperature=TEMP_EXTRACT,
)

print("SHORT SUMMARY")
print("=" * 60)
wrap_print(short_summary)
print(f"\nWord count: {len(short_summary.split())}")

---

# Part C — Tone & Style Controls

## 16. Explain: How Prompt Adaptation Works

The same article produces completely different outputs depending on the **system prompt** and **generation temperature**. Here's the anatomy of prompt adaptation:

| Component | What It Controls | Example |
|-----------|-----------------|--------|
| **System prompt** | Role, format rules, constraints | "You write LinkedIn posts" vs "You write Twitter threads" |
| **Format instructions** | Structure, length, elements | "5-8 tweets, 280 chars each" vs "one paragraph, 1500 chars" |
| **Tone descriptors** | Word choice, sentence structure | "professional" vs "casual" vs "provocative" |
| **Temperature** | Randomness / creativity | 0.3 (predictable) vs 0.7 (creative) |
| **Output format** | JSON vs plain text vs structured | Threads need arrays, emails need key-value |

### Why not one prompt for everything?

A single "convert to social" prompt fails because:
1. Each platform has **different length limits** — the LLM can't optimize for all simultaneously
2. **Tone leakage** — LinkedIn's professional tone bleeds into Twitter's punchy style
3. **Structural mismatch** — threads need numbered items; LinkedIn needs flowing paragraphs
4. **CTA differences** — "Comment below" (LinkedIn) vs "Bookmark this" (Twitter) vs "Read more →" (Email)

## 17. Tone Variations — Same Format, Different Styles

Generate LinkedIn posts in 4 different tones to show how **tone descriptors** in the system prompt change the output.

In [ ]:
TONE_CONFIGS = [
    {
        "name": "Professional / Formal",
        "instruction": (
            "Write in a professional, authoritative tone. Use industry terminology. "
            "No emojis. Third person or first-person plural ('we'). Data-driven. "
            "Suitable for a CTO or VP of Engineering audience."
        ),
        "temperature": TEMP_FORMAL,
    },
    {
        "name": "Casual / Conversational",
        "instruction": (
            "Write like you're telling a friend about your week at a coffee shop. "
            "Use 'I' and 'we'. Contractions are fine. Light humor. Emojis OK (1-2). "
            "Short punchy sentences. Relatable."
        ),
        "temperature": TEMP_CASUAL,
    },
    {
        "name": "Storytelling / Narrative",
        "instruction": (
            "Write as a narrative with a clear arc: problem → struggle → discovery → "
            "result. Open with a vivid scene or moment. Use sensory language. "
            "Build tension before the reveal. Make the reader feel the journey."
        ),
        "temperature": TEMP_GENERATE,
    },
    {
        "name": "Provocative / Hot Take",
        "instruction": (
            "Write a bold, opinionated take. Start with a contrarian statement. "
            "Challenge conventional wisdom. Use strong verbs. Be direct and "
            "slightly confrontational — but back it up with data. Polarizing is OK."
        ),
        "temperature": TEMP_CASUAL,
    },
]

TONE_PROMPT = """Rewrite this article as a LinkedIn post (800-1200 chars).

TONE INSTRUCTIONS:
{tone_instruction}

Article:
{article}

LinkedIn post:"""

print("TONE VARIATIONS — Same Article, 4 Styles")
print("=" * 60)

tone_outputs = {}
for config in TONE_CONFIGS:
    output = ask(
        TONE_PROMPT.format(
            tone_instruction=config["instruction"],
            article=ARTICLE[:2500],  # Shorten for faster generation
        ),
        temperature=config["temperature"],
    )
    tone_outputs[config["name"]] = output

    print(f"\n{'─'*60}")
    print(f"🎨 {config['name']} (temp={config['temperature']})")
    print(f"{'─'*60}")
    wrap_print(output)
    print(f"\n  [{len(output)} chars]")

## 18. Compare Tone Outputs

Analyze how the different tones affected word choice and structure.

In [ ]:
COMPARE_PROMPT = """Compare these 4 LinkedIn posts written in different tones from
the same source article. For each, identify:
- The opening hook strategy
- Word choice characteristics (formal vs casual, jargon level)
- Which audience would respond best
- Engagement prediction (which would get most comments?)

Posts:
{posts}

Analysis:"""

posts_text = "\n\n".join(
    f"--- {name} ---\n{text[:500]}" for name, text in tone_outputs.items()
)

comparison = ask(
    COMPARE_PROMPT.format(posts=posts_text),
    temperature=TEMP_EXTRACT,
)

print("TONE COMPARISON ANALYSIS")
print("=" * 60)
wrap_print(comparison)

## 19. Temperature Experiment

How does **temperature** alone affect the same prompt? Generate 3 versions at different temperatures.

In [ ]:
TEMP_TEST_PROMPT = """Write a 2-sentence LinkedIn hook for this article. 
Make it attention-grabbing.

Article title: {title}
Key stat: Global p99 latency dropped from 340ms to 47ms (86% reduction)

Hook (2 sentences only):"""

title = ARTICLE.split("\n")[0].replace("Title: ", "")

print("TEMPERATURE EXPERIMENT — Same Prompt, Different Temperatures")
print("=" * 60)

for temp in [0.0, 0.5, 1.0]:
    hook = ask(
        TEMP_TEST_PROMPT.format(title=title),
        temperature=temp,
    )
    print(f"\n  🌡 Temperature: {temp}")
    print(f"  {hook}")

print(f"\n{'─'*60}")
print("→ temp=0.0: Most predictable, safest")
print("→ temp=0.5: Balanced creativity")
print("→ temp=1.0: Most creative, sometimes off-brand")

---

# Part D — Advanced Repurposing

## 20. All Formats at Once — Full Repurposing Pipeline

In [ ]:
def repurpose_article(article: str, tone: str = "professional") -> dict:
    """
    Full content repurposing pipeline.
    Takes one article, returns all social formats.
    """
    results = {}

    # 1. Short summary
    results["summary"] = ask(
        f"Write a 2-3 sentence summary of this article. Neutral tone.\n\n{article}",
        temperature=TEMP_EXTRACT,
    )

    # 2. LinkedIn
    results["linkedin"] = ask(
        f"Convert this into a LinkedIn post (1000-1300 chars). "
        f"Tone: {tone}. Include hook, key insight, question at end, 3 hashtags."
        f"\n\n{article}",
        temperature=TEMP_GENERATE,
    )

    # 3. Twitter thread
    thread_raw = ask(
        f"Convert into a Twitter thread of 5-7 tweets. Each ≤ 280 chars. "
        f"Return JSON array: [\"tweet1\", \"tweet2\", ...]. Tone: {tone}."
        f"\n\n{article}",
        temperature=TEMP_GENERATE,
    )
    results["twitter_thread"] = parse_json(thread_raw) or [thread_raw]

    # 4. Email
    email_raw = ask(
        f'Convert into an email snippet. Return JSON: '
        f'{{"subject_line": "...", "preview_text": "...", "body": "...", '
        f'"cta": "Read more →"}}. Body: 100-150 words. Tone: {tone}.'
        f'\n\n{article}',
        temperature=TEMP_GENERATE,
    )
    results["email"] = parse_json(email_raw) or {"body": email_raw}

    return results


print("repurpose_article() pipeline defined")
print("Usage: results = repurpose_article(article_text, tone='casual')")

## 21. Test With a Second Article

Prove the pipeline works on any article, not just the first one.

In [ ]:
ARTICLE_2 = """Title: We Replaced Our Entire Hiring Process With Work Trials — Here's the Data

A year ago, we stopped doing traditional interviews. No whiteboard problems, no
"tell me about a time when" behavioral questions, no take-home assignments that
steal 8 hours of a candidate's weekend. Instead, we pay candidates for a 3-day
work trial where they join our actual team and work on a real (but scoped) project.

The results after 14 months and 47 hires: our 90-day attrition dropped from 24%
to 4%. Our diversity metrics improved — women and underrepresented candidates who
previously dropped out during our 5-round interview process now complete the trial
at the same rate as other candidates. Engineering manager satisfaction with new
hires jumped from 6.2 to 8.8 out of 10.

How it works: candidates get a $1,500 stipend for the 3 days. They're paired with
a team buddy, given access to a sandbox environment, and work on a real ticket from
our backlog (scoped to 2-3 days). We evaluate: code quality, communication,
collaboration, and problem-solving approach — not speed or output volume.

The biggest objection we hear: 'We can't afford to pay every candidate.' But our
old process cost $12,000 per hire in recruiter time, interviewer hours, and lost
productivity. The work trial costs $1,500 per candidate and we interview 60% fewer
people because the trial self-selects. Net savings: $4,200 per hire.

Is this approach for everyone? Probably not. It works best for roles where you can
define a clear, self-contained project. It doesn't work well for executive hires
or roles requiring specialized domain knowledge that takes weeks to onboard. But
for software engineers, designers, and product managers, it's been transformative."""

print("Running full pipeline on Article 2...")
print("=" * 60)

results_2 = repurpose_article(ARTICLE_2, tone="casual")

print(f"\n📝 SUMMARY")
print(f"{'─'*60}")
wrap_print(results_2["summary"])

print(f"\n💼 LINKEDIN")
print(f"{'─'*60}")
wrap_print(results_2["linkedin"])

print(f"\n🐦 TWITTER THREAD")
print(f"{'─'*60}")
if isinstance(results_2["twitter_thread"], list):
    for i, tweet in enumerate(results_2["twitter_thread"], 1):
        print(f"  [{i}] {tweet}")
else:
    print(results_2["twitter_thread"])

print(f"\n📧 EMAIL")
print(f"{'─'*60}")
if isinstance(results_2["email"], dict):
    print(f"  Subject: {results_2['email'].get('subject_line', '')}")
    print(f"  Preview: {results_2['email'].get('preview_text', '')}")
    print()
    wrap_print(results_2["email"].get("body", ""))

## 22. Quality Evaluation — LLM-as-Judge

In [ ]:
JUDGE_SYSTEM = """You evaluate content quality for social media posts.
Score each dimension 1-5. Be critical — most content is a 3."""

JUDGE_PROMPT = """Evaluate this {format} post generated from a blog article.

Original article title: {title}
Generated content:
{content}

Score 1-5 on:
- HOOK: Does the opening grab attention?
- FAITHFULNESS: Does it accurately represent the article?
- PLATFORM_FIT: Does it follow {format} norms and constraints?
- ENGAGEMENT: Would this get likes/comments/shares?

Return ONLY JSON:
{{"hook": N, "faithfulness": N, "platform_fit": N, "engagement": N, 
  "strengths": "...", "improvements": "..."}}"""

formats_to_judge = [
    ("LinkedIn", linkedin_post),
    ("Email", email.get("body", "") if email else ""),
    ("Short Summary", short_summary),
]

print("CONTENT QUALITY EVALUATION")
print("=" * 60)

for fmt, content in formats_to_judge:
    if not content:
        continue
    resp = ask(
        JUDGE_PROMPT.format(
            format=fmt, title=title, content=content[:1000],
        ),
        system=JUDGE_SYSTEM,
        temperature=TEMP_EXTRACT,
    )
    scores = parse_json(resp)
    if scores:
        avg = sum(scores.get(k, 0) for k in ["hook", "faithfulness",
                  "platform_fit", "engagement"]) / 4
        print(f"\n  📊 {fmt}")
        print(f"     Hook: {scores.get('hook',0)}/5  "
              f"Faith: {scores.get('faithfulness',0)}/5  "
              f"Fit: {scores.get('platform_fit',0)}/5  "
              f"Engage: {scores.get('engagement',0)}/5  "
              f"→ Avg: {avg:.1f}/5")
        print(f"     ✅ {scores.get('strengths', '')}")
        print(f"     🔧 {scores.get('improvements', '')}")

---

## 23. Iterative Refinement — Improve Based on Feedback

In [ ]:
REFINE_PROMPT = """Improve this LinkedIn post based on the feedback.

ORIGINAL POST:
{post}

FEEDBACK:
- Make the hook more attention-grabbing (lead with the surprising stat)
- Shorten paragraphs — max 2 lines each
- End with a more specific question, not a generic one
- Keep total length under 1300 characters

IMPROVED POST:"""

refined = ask(
    REFINE_PROMPT.format(post=linkedin_post),
    temperature=TEMP_GENERATE,
)

print("REFINED LINKEDIN POST (v2)")
print("=" * 60)
print(refined)
print(f"\n[{len(refined)} chars]")

print(f"\n{'─'*60}")
print("BEFORE vs AFTER:")
print(f"  v1: {len(linkedin_post):,} chars")
print(f"  v2: {len(refined):,} chars")
print(f"  v1 first line: {linkedin_post.split(chr(10))[0][:70]}...")
print(f"  v2 first line: {refined.split(chr(10))[0][:70]}...")

## 24. Hashtag Generator

In [ ]:
HASHTAG_PROMPT = """Generate hashtags for this article for each platform.

Article title: {title}
Key topics: {topics}

Return ONLY JSON:
{{
  "linkedin": ["#Hashtag1", "#Hashtag2", "#Hashtag3", "#Hashtag4", "#Hashtag5"],
  "twitter": ["#tag1", "#tag2", "#tag3"],
  "instagram": ["#tag1", "#tag2", "#tag3", "#tag4", "#tag5", "#tag6", "#tag7"]
}}

Rules:
- LinkedIn: 3-5 professional hashtags, capitalize words
- Twitter: 1-3 short hashtags (popular ones, lowercase)
- Instagram: 5-10 mix of broad and niche"""

topics = ", ".join(key_points.get("key_points", [])[:4]) if key_points else "edge computing, latency, infrastructure"

hashtag_resp = ask(
    HASHTAG_PROMPT.format(title=title, topics=topics),
    temperature=TEMP_EXTRACT,
)
hashtags = parse_json(hashtag_resp)

print("PLATFORM-SPECIFIC HASHTAGS")
print("=" * 60)

if hashtags:
    for platform in ["linkedin", "twitter", "instagram"]:
        tags = hashtags.get(platform, [])
        icon = {"linkedin": "💼", "twitter": "🐦", "instagram": "📸"}.get(platform, "")
        print(f"\n  {icon} {platform.title()}: {' '.join(tags)}")
else:
    print(hashtag_resp[:200])

---

## 25. Prompt Adaptation Explained

### How Each Format Prompt Differs

Here's a side-by-side breakdown of what makes each prompt unique:

| Element | LinkedIn | Twitter/X | Email | Summary |
|---------|----------|-----------|-------|---------|
| **Length** | 1000-1500 chars | 280 chars/tweet | 100-150 words body | 2-3 sentences |
| **Structure** | Flowing paragraphs, line breaks | Numbered tweets, one point each | Subject + preview + body + CTA | Single paragraph |
| **Hook** | Personal story or bold statement | Surprising stat or provocative claim | Curiosity-driven subject line | None (neutral) |
| **Tone** | Professional but human | Punchy, direct | Conversational, friendly | Neutral, informational |
| **CTA** | "What do you think?" question | "Bookmark / RT / Follow" | "Read the full post →" | None |
| **Emojis** | Sparingly (0-3) | Optional (0-2) | None | None |
| **Hashtags** | 3-5 at end | 1-3 inline or end | None | None |
| **First person** | Yes ("I" or "we") | Yes ("I") | Yes ("I") | No (third person) |
| **Temperature** | 0.5 (balanced) | 0.5 (balanced) | 0.5 (balanced) | 0.0 (deterministic) |

### The Prompt Adaptation Formula

Every format-specific prompt follows this structure:

```
SYSTEM: You are a {platform} content writer.
  - Rule 1: {length constraint}
  - Rule 2: {structural rule}
  - Rule 3: {tone instruction}
  - Rule 4: {engagement element}

USER: Convert this article into a {format}.
  Tone: {tone_modifier}
  Article: {source_text}
  Output format: {json_or_plaintext}
```

The **system prompt** defines the role and rules. The **user prompt** provides the content and format. The **temperature** controls creativity vs consistency.

## 26. Common Mistakes

| Mistake | Why It's Wrong | What to Do Instead |
|---------|---------------|-------------------|
| **One prompt for all formats** | LLM can't optimize for conflicting constraints simultaneously | Use separate, format-specific prompts |
| **No length enforcement** | LinkedIn posts that are 3000 chars, tweets that are 400 chars | State exact char limits in the prompt + validate output |
| **Ignoring platform culture** | A LinkedIn post that reads like a tweet looks tone-deaf | Study each platform's norms: hooks, CTAs, formatting |
| **Same temperature everywhere** | Summaries should be deterministic; social posts need creativity | Use 0.0 for extraction/summary, 0.5+ for creative content |
| **No key-point extraction first** | Going directly from article to social post loses nuance | Extract key points first, then generate from structured data |
| **Tone leakage between formats** | Generating all formats in sequence without resetting context | Use independent LLM calls with fresh system prompts |
| **No iterative refinement** | Accepting the first draft as final | Generate → evaluate → refine based on specific feedback |
| **Ignoring the hook** | Burying the interesting part in paragraph 3 | First 2 lines must grab attention (especially LinkedIn and Twitter) |

## 27. Production Improvement Ideas

1. **Brand voice profile** — define a style guide (words to use/avoid, typical sentence length, emoji policy) and inject it into every prompt
2. **A/B test generation** — generate 2 LinkedIn hooks and let the user pick the better one
3. **Scheduling integration** — connect to Buffer/Hootsuite API to auto-schedule generated content
4. **Image prompt generation** — generate a DALL-E / Midjourney prompt for a featured image
5. **Analytics feedback loop** — track which generated posts get the most engagement and fine-tune prompts
6. **Multi-language support** — generate content in target language directly (not translate after)
7. **Content calendar** — given 4 articles per month, create a 30-day posting calendar across platforms
8. **Audience persona injection** — adjust content for "startup founders" vs "enterprise CTOs" vs "indie hackers"

## 28. Exercises

### Exercise 1: Use Your Own Article

In [ ]:
# ── Exercise 1: Paste your own article ────────────────
# Replace MY_ARTICLE with your blog post or article text.

# MY_ARTICLE = """Paste your article here..."""
# results = repurpose_article(MY_ARTICLE, tone="professional")
# print(results["linkedin"])

print("Exercise 1: Paste your own article and run repurpose_article().")
print("Try different tones: 'professional', 'casual', 'provocative', 'educational'")

### Exercise 2: Add a New Format — Instagram Caption

In [ ]:
# ── Exercise 2: Instagram caption format ──────────────

INSTAGRAM_SYSTEM = """You write Instagram captions for a tech/business account.

Rules:
- First line: attention-grabbing hook (emoji + bold statement)
- Body: 3-5 short paragraphs with line breaks
- Include 1-2 relevant emojis per paragraph
- End with a CTA: 'Save this for later' or 'Tag someone who needs this'
- Add 15-20 hashtags at the end (mix of popular and niche)
- Total caption: 150-300 words
- Tone: inspirational, relatable, value-packed"""

INSTAGRAM_PROMPT = """Convert this article into an Instagram caption.

Article:
{article}

Instagram caption:"""

ig_caption = ask(
    INSTAGRAM_PROMPT.format(article=ARTICLE[:2000]),
    system=INSTAGRAM_SYSTEM,
    temperature=TEMP_CASUAL,
)

print("INSTAGRAM CAPTION")
print("=" * 60)
print(ig_caption)
print(f"\n[{len(ig_caption)} chars, {len(ig_caption.split())} words]")

### Exercise 3: Audience-Specific Adaptation

In [ ]:
# ── Exercise 3: Same article, different audiences ─────

AUDIENCE_PROMPT = """Rewrite this article's key message as a LinkedIn post
specifically for this audience:

TARGET AUDIENCE: {audience}
What they care about: {cares_about}

Article:
{article}

LinkedIn post (800-1200 chars):"""

audiences = [
    ("CTO / VP Engineering", "reliability, architecture decisions, team velocity"),
    ("Startup Founder (non-technical)", "growth metrics, cost savings, competitive advantage"),
    ("Junior Developer", "learning opportunities, career growth, technical concepts explained simply"),
]

print("AUDIENCE-SPECIFIC VERSIONS")
print("=" * 60)

for audience, cares in audiences:
    post = ask(
        AUDIENCE_PROMPT.format(
            audience=audience, cares_about=cares,
            article=ARTICLE[:2000],
        ),
        temperature=TEMP_GENERATE,
    )
    print(f"\n{'─'*60}")
    print(f"🎯 Audience: {audience}")
    print(f"{'─'*60}")
    wrap_print(post)
    print(f"  [{len(post)} chars]")

### Mini Challenge

1. **Content scoring pipeline:** For each generated post, compute a "readability score" using Flesch-Kincaid (or approximate it: avg sentence length × avg word length). Which format produces the most readable content? Does tone affect readability?

2. **Cross-platform consistency:** Generate all 4 formats from the same article and ask the LLM: "Do these 4 pieces of content tell a consistent story? Is any piece missing a key point that others include?" Report the consistency score.

3. **Viral hook A/B test:** Generate 5 different hooks for the same article. Have the LLM rank them by predicted engagement. Then ask: what makes the top-ranked hook better than the bottom one? Extract the pattern.

4. **Multi-article series:** Given 3 related articles, generate a LinkedIn "content series" plan: what to post each day, how to cross-reference between posts, and how to build audience anticipation.

## 29. Session Summary

In [ ]:
print("SESSION SUMMARY")
print("=" * 50)

print(f"\nSource articles processed: 2")
print(f"Formats generated:")
print(f"  ✅ LinkedIn post (with 4 tone variations)")
print(f"  ✅ Twitter/X thread (5-8 tweets, 280 char limit)")
print(f"  ✅ Email newsletter snippet (subject + preview + body)")
print(f"  ✅ Short summary (2-3 sentences, neutral)")
print(f"  ✅ Instagram caption (exercise)")

print(f"\nTone variations tested:")
print(f"  🎨 Professional / Formal")
print(f"  🎨 Casual / Conversational")
print(f"  🎨 Storytelling / Narrative")
print(f"  🎨 Provocative / Hot Take")

print(f"\nAdvanced features:")
print(f"  ✅ Key point extraction")
print(f"  ✅ Temperature experiment")
print(f"  ✅ Quality evaluation (LLM-as-judge)")
print(f"  ✅ Iterative refinement")
print(f"  ✅ Hashtag generation")
print(f"  ✅ Audience-specific adaptation")
print(f"  ✅ Prompt adaptation analysis")

## 30. Key Takeaways

| # | Takeaway |
|---|----------|
| 1 | **Separate prompts per format** beat a single "convert to social" prompt — each platform has unique norms |
| 2 | **The system prompt defines the role and rules**; the user prompt provides content and specifics |
| 3 | **Tone is controlled by descriptors** ("professional", "casual") AND temperature (0.3 vs 0.7) |
| 4 | **Extract key points first**, then generate — structured input creates better output |
| 5 | **Always validate constraints** — check tweet lengths, char counts, word counts after generation |
| 6 | **Iterative refinement** (generate → evaluate → refine) produces publication-quality content |
| 7 | **Audience-specific adaptation** changes what you emphasize, not just how you write |
| 8 | **The hook is everything** — on LinkedIn and Twitter, the first 2 lines determine if anyone reads the rest |

---

*This notebook is part of the [Machine Learning Projects](https://github.com/pypi-ahmad/Machine-Learning-Projects) repository.*